In [1]:
import pandas as pd
import scanpy as sc
import numpy as np
import anndata as ad
import os
import colorcet as cc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import harmonypy as hm
import scrublet as scr
import traceback
from collections import Counter
from sklearn.decomposition import PCA
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import re
from matplotlib.patches import Patch
from PIL import Image

# --- ADOBE ILLUSTRATOR COMPATIBILITY ---
# 42 is for TrueType fonts (editable text in Illustrator)
sc.settings.set_figure_params(dpi=300, dpi_save=300)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# --- NOTEBOOK SETTINGS ---
%matplotlib inline

# Define Working Directory
working_dic = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/"
os.chdir(working_dic)

# --- SCANPY SETTINGS ---
sc.settings.verbosity = 1
sc.logging.print_header()

# Professional Plotting Parameters
# sc.set_figure_params( dpi=100,  dpi_save=300,  vector_friendly=True, format='pdf',  transparent=True, facecolor='white' )

# --- CUSTOM COLOR SCHEMES ---
# Your grey-to-blue gradient for feature plots
dimplot_color = mcolors.LinearSegmentedColormap.from_list("custom_cmap", ["#E5E5E5", "#4C4DB7"])
COND_ORDER = [
    "Control",
    "Prematurely-terminated",
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
]

COND_COLORS = {
    "Control": "#EFE48E",
    "3month-protocol": "skyblue",
    "6month-protocol": "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV": "#F0C096",
    "Rejection": "#E09E9E",
}


In [2]:
global_raw = sc.read ('/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/global_object_assembly/objects/global_X_UMAP_Added_PCMV_score_updated_celltype_cellstate_annotated_preclinical_object_RAW_v3.h5ad')


In [3]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix, hstack
from tqdm import tqdm

def merge_genes_by_prefix(adata, prefixes):
    """
    Efficiently merges genes by removing species prefixes,
    preserving metadata, and ensuring Seurat/CellChat compatibility.
    """
    # 1. Create clean gene names
    gene_series = pd.Series(adata.var_names)
    for prefix in prefixes:
        gene_series = gene_series.str.replace(prefix, "", regex=False)
    
    # Update temporary metadata
    adata.var["gene_short"] = gene_series.values
    
    # 2. Prepare new var dataframe (keeping original metadata)
    # Use 'first' to keep original annotations for the merged genes
    new_var = adata.var.groupby("gene_short").first()
    
    # 3. Summing logic (Optimized for memory)
    gene_groups = adata.var.groupby("gene_short").groups
    summed_matrices = []
    
    # Ensure X is CSR for column slicing
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
        
    for gene in tqdm(new_var.index, desc="Merging genes"):
        indices = gene_groups[gene]
        # Map to integer column locations
        col_indices = [adata.var_names.get_loc(i) for i in indices]
        
        # Sum columns; cast to integer to maintain count integrity
        summed = adata.X[:, col_indices].sum(axis=1)
        summed_matrices.append(csr_matrix(summed).astype(np.int32))
        
    # 4. Reconstruct AnnData
    merged_X = hstack(summed_matrices, format="csr")
    
    adata_merged = sc.AnnData(
        X=merged_X,
        obs=adata.obs.copy(),
        var=new_var, # Metadata preserved
        uns=adata.uns.copy(),
        obsm=adata.obsm.copy()
    )
    
    return adata_merged

# --- Execution ---
prefixes = ["pCMV_____", "Papio_anubis_", "Sus_scrofa___"]
adata_merged = merge_genes_by_prefix(global_raw, prefixes)

# Final safety check before saving
if len(adata_merged.var_names) != len(adata_merged.var_names.unique()):
    print("Warning: Duplicate gene names found after merge!")

# Save path
output_path = '/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/Raw_UMI_merged_global_object_preclinical.h5ad'
adata_merged.write(output_path)

Merging genes: 100%|██████████| 40653/40653 [6:05:52<00:00,  1.85it/s]  
/data/ag_reichart_lindberg/nhadizad/miniconda3/envs/amylo/lib/python3.11/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


In [2]:
umi_merged_object = sc.read ('/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/Raw_UMI_merged_global_object_preclinical.h5ad')

In [3]:
umi_merged_object

AnnData object with n_obs × n_vars = 233200 × 40653
    obs: 'Sample_ID', 'source', 'doublet_score', 'predicted_doublet_scrublet', 'n_counts_Sus_scrofa', 'n_counts_Papio_anubis', 'species_origin', 'barnyard_doublet', 'Survival', 'Condition', 'Donor_heart_mass (g)', 'Recipient_heart_mass (g)', 'Recipient_LVEF (%)', 'Recipient_LV_Wall_Thickness (mm)', 'TropT (ng/ml)', 'Immunosuppression_Protocol', 'Heart_Mass_Ratio', 'Exp', 'percent_mito', 'percent_ribo', 'n_genes_by_counts', 'total_counts', 'n_counts', 'n_genes', 'n_counts_PCMV', 'n_genes_PCMV', 'PCMV_score', 'cell_type', 'cell_state', 'n_counts_pCMV', 'n_genes_pCMV', 'pCMV_score', 'PCMV_detection_threshold'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'Condition_colors', 'Sample_ID_colors', 'cell_state_colors', 'cell_type_colors'
    obsm: 'X_pca_harmony', 'X_umap', 'X_umap_CM', 'X_umap_EC', 'X_umap_FB', 'X_umap_Lymph_papio', 'X_umap_Lymph_sus', 'X_

In [4]:
import scipy.io
import os

# Create a folder to hold the transfer files
out_dir = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/seurat_transfer/"
os.makedirs(out_dir, exist_ok=True)

# 1. Export the sparse matrix (Transpose it because Seurat expects Genes x Cells)
scipy.io.mmwrite(out_dir + "matrix.mtx", umi_merged_object.X.T)

# 2. Export barcodes (cells) and features (genes)
umi_merged_object.obs_names.to_frame().to_csv(out_dir + "barcodes.tsv", index=False, header=False)
umi_merged_object.var_names.to_frame().to_csv(out_dir + "features.tsv", index=False, header=False)

# 3. Export metadata
umi_merged_object.obs.to_csv(out_dir + "metadata.csv")

print("Export complete! Ready for R.")

Export complete! Ready for R.
